In [12]:
import pandas as pd
import json
import os

print("Libraries imported successfully.")

Libraries imported successfully.


In [13]:
#Load Memory Tables

short_term_path = "../data/processed/short_term_memory.csv"
long_term_path = "../data/processed/long_term_memory.csv"
concept_memory_path = "../data/processed/concept_based_memory.csv"

short_term_df = pd.read_csv(short_term_path)
long_term_df = pd.read_csv(long_term_path)
concept_memory_df = pd.read_csv(concept_memory_path)

print("Memory tables loaded successfully.")
print("Short-term memory shape:", short_term_df.shape)
print("Long-term memory shape:", long_term_df.shape)
print("Concept-based memory shape:", concept_memory_df.shape)

Memory tables loaded successfully.
Short-term memory shape: (4151, 8)
Long-term memory shape: (4151, 12)
Concept-based memory shape: (39309, 12)


In [14]:
#Check Missing Values

print("Short-Term Memory Missing Values")
print(short_term_df.isnull().sum())

print("\nLong-Term Memory Missing Values")
print(long_term_df.isnull().sum())

print("\nConcept-Based Memory Missing Values")
print(concept_memory_df.isnull().sum())

Short-Term Memory Missing Values
student_id                  0
session_id                  0
current_concept             0
recent_interaction_count    0
recent_accuracy             0
average_attempts            0
recent_hint_usage           0
session_status              0
dtype: int64

Long-Term Memory Missing Values
student_id                  0
total_interactions          0
total_sessions              0
total_concepts              0
overall_accuracy            0
average_attempts            0
average_hint_count          0
total_hint_count            0
average_response_time_ms    0
weak_areas                  0
strong_areas                0
preferred_support_style     0
dtype: int64

Concept-Based Memory Missing Values
student_id              0
concept_name            0
total_interactions      0
correct_count           0
avg_hints               0
total_hints             0
avg_attempts            0
avg_response_time_ms    0
wrong_count             0
accuracy                0
observed_be

In [15]:
#Validate Student Coverage

#This proves all three memory layers are connected.

short_students = set(short_term_df["student_id"].unique())
long_students = set(long_term_df["student_id"].unique())
concept_students = set(concept_memory_df["student_id"].unique())

print("Students in short-term memory:", len(short_students))
print("Students in long-term memory:", len(long_students))
print("Students in concept-based memory:", len(concept_students))

common_students = short_students.intersection(long_students).intersection(concept_students)

print("Students available in all memory layers:", len(common_students))

Students in short-term memory: 4151
Students in long-term memory: 4151
Students in concept-based memory: 4151
Students available in all memory layers: 4151


In [16]:
#Validate Short-Term Memory

#This shows how many students are currently in:

short_term_summary = short_term_df["session_status"].value_counts()

print("Short-term session status distribution:")
print(short_term_summary)

short_term_summary_df = short_term_summary.reset_index()
short_term_summary_df.columns = ["session_status", "student_count"]

short_term_summary_df

Short-term session status distribution:
session_status
good_progress        1763
needs_support        1602
moderate_progress     786
Name: count, dtype: int64


,session_status,student_count
0,good_progress,1763
1,needs_support,1602
2,moderate_progress,786


In [17]:
#Validate Long-Term Memory

#This supports your claim that long-term memory stores student learning behavior.

support_style_summary = long_term_df["preferred_support_style"].value_counts()

print("Preferred support style distribution:")
print(support_style_summary)

support_style_summary_df = support_style_summary.reset_index()
support_style_summary_df.columns = ["preferred_support_style", "student_count"]

support_style_summary_df

Preferred support style distribution:
preferred_support_style
basic_explanation_support    1801
independent_practice         1499
step_by_step_support          638
guided_support                213
Name: count, dtype: int64


,preferred_support_style,student_count
0,basic_explanation_support,1801
1,independent_practice,1499
2,step_by_step_support,638
3,guided_support,213


In [18]:
#Validate Concept-Based Memory

#This shows concept-level storage quality.

concept_summary = concept_memory_df.groupby("concept_name").agg(
    student_count=("student_id", "nunique"),
    total_records=("student_id", "count"),
    avg_accuracy=("accuracy", "mean"),
    avg_hints=("avg_hints", "mean"),
    avg_attempts=("avg_attempts", "mean")
).reset_index()

concept_summary["avg_accuracy"] = concept_summary["avg_accuracy"].round(2)
concept_summary["avg_hints"] = concept_summary["avg_hints"].round(2)
concept_summary["avg_attempts"] = concept_summary["avg_attempts"].round(2)

concept_summary = concept_summary.sort_values("total_records", ascending=False)

concept_summary.head(10)

,concept_name,student_count,total_records,avg_accuracy,avg_hints,avg_attempts
2,Addition and Subtraction Fractions,1353,1353,0.72,0.61,1.95
3,Addition and Subtraction Integers,1226,1226,0.75,0.27,2.21
23,Conversion of Fraction Decimals Percents,1225,1225,0.68,0.62,3.36
67,Percent Of,1115,1115,0.58,0.78,4.11
0,Absolute Value,1002,1002,0.79,0.03,1.04
1,Addition Whole Numbers,988,988,0.78,0.09,1.19
31,Equation Solving Two or Fewer Steps,961,961,0.65,0.40,2.35
62,Ordering Positive Decimals,942,942,0.81,0.07,1.23
73,Probability of a Single Event,939,939,0.79,0.24,1.29
96,Subtraction Whole Numbers,903,903,0.70,0.10,1.00


In [19]:
#Create Clean Memory Context Function
#This function should avoid analytical fields like observed_behavior and memory_note.

def get_clean_memory_context(student_id, target_concept=None):
    short_term = short_term_df[short_term_df["student_id"] == student_id]
    long_term = long_term_df[long_term_df["student_id"] == student_id]
    concept_records = concept_memory_df[concept_memory_df["student_id"] == student_id]

    if short_term.empty:
        return {"error": "Student not found in short-term memory"}

    if long_term.empty:
        return {"error": "Student not found in long-term memory"}

    if concept_records.empty:
        return {"error": "Student not found in concept-based memory"}

    short_term_record = short_term.iloc[0].to_dict()
    long_term_record = long_term.iloc[0].to_dict()

    if target_concept is None:
        target_concept = concept_records.sort_values(
            by="total_interactions",
            ascending=False
        ).iloc[0]["concept_name"]

    target_concept_records = concept_records[
        concept_records["concept_name"] == target_concept
    ]

    if target_concept_records.empty:
        concept_record = {
            "concept_name": target_concept,
            "message": "No concept-level interaction history found for this student and concept."
        }
    else:
        concept_raw = target_concept_records.iloc[0].to_dict()

        concept_record = {
            "concept_name": concept_raw.get("concept_name"),
            "total_interactions": int(concept_raw.get("total_interactions", 0)),
            "correct_count": int(concept_raw.get("correct_count", 0)),
            "wrong_count": int(concept_raw.get("wrong_count", 0)),
            "accuracy": float(concept_raw.get("accuracy", 0)),
            "total_hints": int(concept_raw.get("total_hints", 0)),
            "avg_hints": float(concept_raw.get("avg_hints", 0)),
            "avg_attempts": float(concept_raw.get("avg_attempts", 0)),
            "avg_response_time_ms": float(concept_raw.get("avg_response_time_ms", 0)),
            "last_interaction_order": int(concept_raw.get("last_interaction_order", 0))
        }

    memory_context = {
        "student_id": int(student_id),
        "target_concept": target_concept,
        "short_term_memory": {
            "session_id": int(short_term_record["session_id"]),
            "current_concept": short_term_record["current_concept"],
            "recent_interaction_count": int(short_term_record["recent_interaction_count"]),
            "recent_accuracy": float(short_term_record["recent_accuracy"]),
            "average_attempts": float(short_term_record["average_attempts"]),
            "recent_hint_usage": int(short_term_record["recent_hint_usage"]),
            "session_status": short_term_record["session_status"]
        },
        "long_term_memory": {
            "total_interactions": int(long_term_record["total_interactions"]),
            "total_sessions": int(long_term_record["total_sessions"]),
            "total_concepts": int(long_term_record["total_concepts"]),
            "overall_accuracy": float(long_term_record["overall_accuracy"]),
            "average_attempts": float(long_term_record["average_attempts"]),
            "average_hint_count": float(long_term_record["average_hint_count"]),
            "total_hint_count": int(long_term_record["total_hint_count"]),
            "average_response_time_ms": float(long_term_record["average_response_time_ms"]),
            "preferred_support_style": long_term_record["preferred_support_style"]
        },
        "concept_based_memory": concept_record,
        "integration_note": {
            "for_meta_agent": "Use this stored interaction history to derive mastery, knowledge graph updates, regression flags, and learning paths.",
            "for_fapr_lb": "Use this memory context as input features for struggle prediction and repair strategy selection.",
            "for_tutor_agent": "Use this context to personalize explanations without treating the student as new."
        }
    }

    return memory_context

In [20]:
#Test Memory Context for One Student

sample_student_id = short_term_df["student_id"].iloc[0]

sample_context = get_clean_memory_context(sample_student_id)

print(json.dumps(sample_context, indent=4, default=str))

{
    "student_id": 74864,
    "target_concept": "Addition and Subtraction Positive Decimals",
    "short_term_memory": {
        "session_id": 261340,
        "current_concept": "Addition and Subtraction Positive Decimals",
        "recent_interaction_count": 1,
        "recent_accuracy": 1.0,
        "average_attempts": 1.0,
        "recent_hint_usage": 0,
        "session_status": "good_progress"
    },
    "long_term_memory": {
        "total_interactions": 1,
        "total_sessions": 1,
        "total_concepts": 1,
        "overall_accuracy": 1.0,
        "average_attempts": 1.0,
        "average_hint_count": 0.0,
        "total_hint_count": 0,
        "average_response_time_ms": 18720.0,
        "preferred_support_style": "independent_practice"
    },
    "concept_based_memory": {
        "concept_name": "Addition and Subtraction Positive Decimals",
        "total_interactions": 1,
        "correct_count": 1,
        "wrong_count": 0,
        "accuracy": 1.0,
        "total_hint

In [21]:
#Compare Two Students

#This proves that different students get different memory contexts.
student_a = short_term_df["student_id"].iloc[0]
student_b = short_term_df["student_id"].iloc[10]

context_a = get_clean_memory_context(student_a)
context_b = get_clean_memory_context(student_b)

comparison = pd.DataFrame([
    {
        "student_id": student_a,
        "session_status": context_a["short_term_memory"]["session_status"],
        "overall_accuracy": context_a["long_term_memory"]["overall_accuracy"],
        "avg_hints": context_a["long_term_memory"]["average_hint_count"],
        "preferred_support_style": context_a["long_term_memory"]["preferred_support_style"],
        "target_concept": context_a["target_concept"]
    },
    {
        "student_id": student_b,
        "session_status": context_b["short_term_memory"]["session_status"],
        "overall_accuracy": context_b["long_term_memory"]["overall_accuracy"],
        "avg_hints": context_b["long_term_memory"]["average_hint_count"],
        "preferred_support_style": context_b["long_term_memory"]["preferred_support_style"],
        "target_concept": context_b["target_concept"]
    }
])

comparison

,student_id,session_status,overall_accuracy,avg_hints,preferred_support_style,target_concept
0,74864,good_progress,1.0,0.0,independent_practice,Addition and Subtraction Positive Decimals
1,79285,needs_support,0.0,0.0,basic_explanation_support,Pythagorean Theorem


In [22]:
#Save Evaluation Outputs

os.makedirs("../outputs/tables", exist_ok=True)
os.makedirs("../outputs/api_results", exist_ok=True)

short_term_summary_df.to_csv(
    "../outputs/tables/short_term_memory_evaluation.csv",
    index=False
)

support_style_summary_df.to_csv(
    "../outputs/tables/long_term_memory_evaluation.csv",
    index=False
)

concept_summary.head(20).to_csv(
    "../outputs/tables/concept_memory_evaluation.csv",
    index=False
)

comparison.to_csv(
    "../outputs/tables/two_student_memory_context_comparison.csv",
    index=False
)

with open("../outputs/api_results/clean_sample_memory_context.json", "w") as f:
    json.dump(sample_context, f, indent=4, default=str)

print("Memory evaluation outputs saved successfully.")

Memory evaluation outputs saved successfully.
